In [13]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

SEED = 42
TEST_SIZE = 0.2
DATA = "../data/train.csv"
DROP = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df = pd.read_csv(DATA).drop(columns=DROP)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

In [14]:
X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])
cat_cols = ["Sex", "Embarked"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool  = Pool(X_test,  y_test,  cat_features=cat_cols)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=100,                 
    early_stopping_rounds=50,    
)
model.fit(train_pool, eval_set=test_pool, use_best_model=True)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("Test AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("Best iteration:", model.get_best_iteration())

0:	test: 0.8317523	best: 0.8317523 (0)	total: 13.7ms	remaining: 13.7s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8567193676
bestIteration = 34

Shrink model to first 35 iterations.
              precision    recall  f1-score   support

           0      0.794     0.909     0.847       110
           1      0.811     0.623     0.705        69

    accuracy                          0.799       179
   macro avg      0.802     0.766     0.776       179
weighted avg      0.800     0.799     0.793       179

Test AUC: 0.8567
Best iteration: 34


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
from utils import log_result

log_result(
    model_name="Catboost",
    y_true=y_test,
    y_proba=y_proba,      
    y_pred=y_pred,       
    params={"learning_rate": "0.05", "eval_metric": "AUC", "loss_function": "Logloss"},
    notes=""
)

✓ Catboost: AUC=0.8567, acc=0.7989 → c:\Users\АуфПК\VSCode\titanic\results.csv


{'model': 'Catboost',
 'timestamp': '2026-11-03 00:28',
 'auc': 0.8567,
 'accuracy': 0.7989,
 'f1': 0.7049,
 'precision': 0.8113,
 'recall': 0.6232,
 'params': '{}',
 'notes': '',
 'notebook': ''}